# CSSF(QA) — D-Wave Pegasus P16 Scalable Evidence Program v60

**Reproducible scientific notebook for CSSF quantum-annealing trigonometrization across four application tasks.**

This notebook executes the validated CSSF(QA) evidence program on Pegasus-P16 simulator infrastructure. It presents the scientific mechanism, matched experimental design, analytical preflight, simulator evidence, physical endpoints, and fail-closed claim gates. GPU/SQA results are not represented as real-QPU evidence.

## Execution environment

Google Colab with an NVIDIA CUDA GPU is required for the production SQA stages. Before any annealer call, the notebook runs a CPU-only Analytical Preflight that checks schedule admissibility, CSSF information geometry, causal completeness, QUBO consistency, and statistical resolution. The analytical layer predicts experiment identifiability and failure risk but never substitutes for GPU SQA evidence.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 02 — HARD GPU RUNTIME GATE
# Must run BEFORE any package installation.
# CPU execution is prohibited for full_cssf_qa_gpu.

import os
import subprocess


print("=" * 80)
print("CSSF-QA — GPU RUNTIME PRECHECK")
print("=" * 80)


# ---------------------------------------------------------------------
# Check that an NVIDIA GPU is physically exposed to the Colab runtime.
# This check intentionally does not depend on project Python packages.
# ---------------------------------------------------------------------

try:
    result = subprocess.run(
        ["nvidia-smi"],
        text=True,
        capture_output=True,
    )
except FileNotFoundError as exc:
    raise RuntimeError(
        "nvidia-smi is unavailable; this is not a supported NVIDIA CUDA Colab runtime. "
        "CPU fallback is prohibited for full_cssf_qa_gpu."
    ) from exc


if result.returncode != 0:
    raise RuntimeError(
        "NVIDIA GPU is not available in this Colab runtime.\n\n"
        "Select a GPU runtime before installing any project dependencies.\n"
        "Colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU\n\n"
        "CPU fallback is prohibited for full_cssf_qa_gpu."
    )


print(result.stdout)


# ---------------------------------------------------------------------
# CUDA device exposure
# ---------------------------------------------------------------------

cuda_visible_devices = os.environ.get(
    "CUDA_VISIBLE_DEVICES",
    "<not explicitly set>",
)

print()
print(f"CUDA_VISIBLE_DEVICES: {cuda_visible_devices}")


# ---------------------------------------------------------------------
# Optional PyTorch CUDA check.
# Colab normally provides PyTorch before project installation.
# ---------------------------------------------------------------------

try:
    import torch
except Exception as exc:
    raise RuntimeError(
        "PyTorch cannot be imported in the base Colab runtime."
    ) from exc


print(f"PyTorch version : {torch.__version__}")
print(f"PyTorch CUDA    : {torch.version.cuda}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"CUDA devices    : {torch.cuda.device_count()}")


if not torch.cuda.is_available():
    raise RuntimeError(
        "NVIDIA hardware is visible, but PyTorch CUDA is unavailable. "
        "Do not install the CSSF-QA environment in this runtime."
    )


if torch.cuda.device_count() < 1:
    raise RuntimeError(
        "No CUDA device is exposed to PyTorch."
    )


device_name = torch.cuda.get_device_name(0)
device_capability = torch.cuda.get_device_capability(0)

print(f"GPU device      : {device_name}")
print(f"GPU capability  : {device_capability}")


# ---------------------------------------------------------------------
# Minimal float64 CUDA execution probe
# ---------------------------------------------------------------------

x = torch.tensor(
    [1.0, 2.0, 3.0],
    dtype=torch.float64,
    device="cuda",
)

probe = (x * x).sum()

if not torch.isfinite(probe):
    raise RuntimeError(
        "CUDA float64 execution probe failed."
    )


print(f"CUDA float64 probe: PASS ({probe.item()})")

print()
print("=" * 80)
print("GPU RUNTIME PRECHECK: PASS")
print("=" * 80)
print()
print("The runtime is suitable for CSSF-QA GPU installation.")

In [ ]:
# 03 — Google Colab compatibility layer; restart the session after this cell
from importlib import metadata
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT_BOOTSTRAP = Path('/content/drive/MyDrive/cssf_dwave')
if not PROJECT_ROOT_BOOTSTRAP.is_dir():
    raise FileNotFoundError(
        f'CSSF(QA) project root is missing: {PROJECT_ROOT_BOOTSTRAP}. '
        'Extract the cumulative release to /content/drive/MyDrive/cssf_dwave before installation.'
    )
if str(PROJECT_ROOT_BOOTSTRAP) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_BOOTSTRAP))

print('=' * 80)
print('CSSF-QA — COLAB COMPATIBILITY INSTALLATION v51')
print('=' * 80)
print('A manual runtime restart is required after successful completion.')
print()

# Upgrade pip first, then make the small packaging library available before
# importing the environment-policy helper.  No scientific package is imported here.
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--progress-bar', 'off', 'pip'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--progress-bar', 'off', 'packaging>=25,<27'],
    check=True,
)

from experiments_dwave.colab_environment_v51 import (
    google_colab_requirements,
    resolve_colab_pandas_install_requirement,
)

COLAB_REQUIREMENTS = google_colab_requirements()
PANDAS_RUNTIME_REQUIREMENT = resolve_colab_pandas_install_requirement(COLAB_REQUIREMENTS)
BASE_REQUIREMENTS = (
    'numpy==2.0.2',
    PANDAS_RUNTIME_REQUIREMENT,
)
OPTIONAL_EDITOR_REQUIREMENTS = (
    'jedi>=0.19,<0.20',
)

print(f'google-colab pandas requirement resolved to: {PANDAS_RUNTIME_REQUIREMENT}')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--progress-bar', 'off', *BASE_REQUIREMENTS],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--progress-bar', 'off', *OPTIONAL_EDITOR_REQUIREMENTS],
    check=False,
)

print()
print('=' * 80)
print('COLAB COMPATIBILITY INSTALLATION COMPLETED SUCCESSFULLY')
print('=' * 80)
print('>>> MANUAL RESTART REQUIRED <<<')
print('Colab menu: Runtime -> Restart session')
print('After the restart, continue with the NEXT notebook cell.')

In [ ]:
# 04 — CSSF(QA) scientific dependencies; continue here after the manual restart
from importlib import metadata
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT_BOOTSTRAP = Path('/content/drive/MyDrive/cssf_dwave')
if not PROJECT_ROOT_BOOTSTRAP.is_dir():
    raise FileNotFoundError(PROJECT_ROOT_BOOTSTRAP)
if str(PROJECT_ROOT_BOOTSTRAP) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_BOOTSTRAP))

from experiments_dwave.colab_environment_v51 import (
    collect_versions,
    google_colab_requirements,
    resolve_colab_pandas_install_requirement,
    validate_base_versions,
)

COLAB_REQUIREMENTS = google_colab_requirements()
PANDAS_RUNTIME_REQUIREMENT = resolve_colab_pandas_install_requirement(COLAB_REQUIREMENTS)
BASE_VERSION_NAMES = ('numpy', 'pandas', 'packaging')
BASE_VERSIONS = collect_versions(BASE_VERSION_NAMES)
base_errors = validate_base_versions(BASE_VERSIONS, COLAB_REQUIREMENTS)
if base_errors:
    raise RuntimeError(
        'The post-restart Colab compatibility layer is not active:\n  - '
        + '\n  - '.join(base_errors)
        + '\nRun the v51 compatibility-installation cell again and restart the session.'
    )

PROJECT_REQUIREMENTS = (
    'numpy==2.0.2',
    PANDAS_RUNTIME_REQUIREMENT,
    'packaging>=25,<27',
    'qiskit==1.2.4',
    'qiskit-algorithms==0.3.1',
    'dwave-ocean-sdk==9.4.0',
    'pandapower==3.2.2',
    'highspy==1.15.1',
    'pyomo>=6.9,<7',
    'scipy>=1.13,<2',
    'networkx>=3.2,<4',
    'scikit-learn>=1.5,<2',
    'joblib>=1.4,<2',
    'tqdm>=4.67,<5',
    'PyYAML>=6.0.2,<7',
    'pydantic>=2.8,<3',
    'typing-extensions>=4.12,<5',
    'matplotlib>=3.9,<4',
    'pyarrow>=17,<23',
)
AER_GPU_REQUIREMENT = 'qiskit-aer-gpu==0.15.1'
AER_GPU_INSTALL_REQUIREMENTS = (
    'numpy==2.0.2',
    PANDAS_RUNTIME_REQUIREMENT,
    'qiskit==1.2.4',
    AER_GPU_REQUIREMENT,
)

print('=' * 80)
print('CSSF-QA — PROJECT DEPENDENCIES v51')
print('=' * 80)
print(f'Preserving Colab host pandas pin: {PANDAS_RUNTIME_REQUIREMENT}')

# qiskit-aer-gpu is the declared GPU distribution.  Remove any CPU/CUDA-11
# Aer wheel first and reinstall the pinned CUDA-12 GPU wheel immediately after.
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'qiskit-aer', 'qiskit-aer-gpu-cu11'],
    check=False,
    text=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--progress-bar', 'off', *PROJECT_REQUIREMENTS],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--upgrade', '--progress-bar', 'off', *AER_GPU_INSTALL_REQUIREMENTS],
    check=True,
)
print('PROJECT DEPENDENCIES INSTALLED SUCCESSFULLY')

In [ ]:
# 05 — post-installation environment verification
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT_BOOTSTRAP = Path('/content/drive/MyDrive/cssf_dwave')
if str(PROJECT_ROOT_BOOTSTRAP) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_BOOTSTRAP))

from experiments_dwave.colab_environment_v51 import (
    FULL_EXACT_VERSIONS,
    PACKAGING_TESTED_SPEC,
    PANDAS_TESTED_SPEC,
    collect_versions,
    google_colab_requirements,
    pip_check_failure,
    resolve_colab_pandas_install_requirement,
    validate_full_versions,
)

COLAB_REQUIREMENTS = google_colab_requirements()
PANDAS_RUNTIME_REQUIREMENT = resolve_colab_pandas_install_requirement(COLAB_REQUIREMENTS)
VERSION_NAMES = tuple(dict.fromkeys((*FULL_EXACT_VERSIONS.keys(), 'pandas', 'packaging', 'jedi')))
VERSIONS = collect_versions(VERSION_NAMES)

version_errors = validate_full_versions(VERSIONS, COLAB_REQUIREMENTS)
for package, expected in FULL_EXACT_VERSIONS.items():
    actual = VERSIONS.get(package)
    status = 'OK' if actual == expected else 'ERROR'
    print(f'{status:5s} {package:24s} expected={expected:18s} actual={actual}')

pandas_actual = VERSIONS.get('pandas')
pandas_ok = not any(e.startswith('pandas:') or e.startswith('pandas does not satisfy') for e in version_errors)
print(
    f'{"OK" if pandas_ok else "ERROR":5s} {"pandas":24s} '
    f'expected={PANDAS_TESTED_SPEC + " & " + PANDAS_RUNTIME_REQUIREMENT:18s} actual={pandas_actual}'
)
packaging_actual = VERSIONS.get('packaging')
packaging_ok = not any(e.startswith('packaging:') for e in version_errors)
print(
    f'{"OK" if packaging_ok else "ERROR":5s} {"packaging":24s} '
    f'expected={PACKAGING_TESTED_SPEC:18s} actual={packaging_actual}'
)
print(f'INFO  {"jedi":24s} optional editor helper actual={VERSIONS.get("jedi")}')

pip_check = subprocess.run(
    [sys.executable, '-m', 'pip', 'check'],
    text=True,
    capture_output=True,
)
if pip_check.stdout:
    print(pip_check.stdout, end='' if pip_check.stdout.endswith('\n') else '\n')
else:
    print('<pip check produced no stdout>')
if pip_check.stderr:
    print(pip_check.stderr, end='' if pip_check.stderr.endswith('\n') else '\n')
pip_error = pip_check_failure(pip_check.returncode, pip_check.stdout, pip_check.stderr)
if pip_error is not None:
    version_errors.append(pip_error)

if version_errors:
    raise RuntimeError('Environment verification failed:\n  - ' + '\n  - '.join(version_errors))

print('ENVIRONMENT VERIFICATION: PASS')
print({'resolved_colab_pandas_requirement': PANDAS_RUNTIME_REQUIREMENT, 'versions': VERSIONS})

In [ ]:
# 06 — D-Wave Pegasus P16 simulator scientific context
from pathlib import Path
import sys

PROJECT_ROOT = Path('/content/drive/MyDrive/cssf_dwave')
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_PATH = PROJECT_ROOT / 'notebooks' / 'CSSF_QA_DWave_Evidence_Simulator_v56.ipynb'
F0_MANIFEST_PATH = PROJECT_ROOT / 'releases' / 'scientific_manifests' / 'FROZEN_SOURCE_MANIFEST_v51.json'
F1_MANIFEST_PATH = PROJECT_ROOT / 'releases' / 'scientific_manifests' / 'FROZEN_INPUT_MANIFEST_v51.json'
RESULTS_ROOT = PROJECT_ROOT / 'results' / 'dwave_evidence_v60' / 'simulator'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / 'dwave_evidence_v60' / 'simulator'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
CALIBRATION_FAMILY = 'Advantage_system6'
if CALIBRATION_FAMILY not in {'Advantage_system4','Advantage_system6'}:
    raise RuntimeError('Only Advantage_system4 or Advantage_system6 are admissible.')
print({'project_root':str(PROJECT_ROOT),'backend':'Pegasus-P16 calibration-resolved CUDA SQA','calibration_family':CALIBRATION_FAMILY})

In [ ]:
# 08 — CSSF source-integrity and framework-identity gate
import hashlib
import json
from pathlib import Path

from experiments_dwave.evidence_program_v53 import framework_identity_report

F0_MANIFEST_PATH = (
    PROJECT_ROOT
    / "releases"
    / "scientific_manifests"
    / "FROZEN_SOURCE_MANIFEST_v51.json"
)

if not F0_MANIFEST_PATH.is_file():
    raise FileNotFoundError(F0_MANIFEST_PATH)
if not NOTEBOOK_PATH.is_file():
    raise FileNotFoundError(NOTEBOOK_PATH)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()

f0_manifest = json.loads(F0_MANIFEST_PATH.read_text(encoding="utf-8"))
if int(f0_manifest.get("count", -1)) != 67:
    raise RuntimeError("The active frozen-source manifest must contain exactly 67 retained original scientific Python files.")

f0_errors = []
for relative_path, expected_sha256 in sorted(f0_manifest["files"].items()):
    source_path = PROJECT_ROOT / relative_path
    if not source_path.is_file():
        f0_errors.append(f"missing: {relative_path}")
        continue
    actual_sha256 = sha256_file(source_path)
    if actual_sha256 != expected_sha256:
        f0_errors.append(
            f"hash mismatch: {relative_path}: {actual_sha256} != {expected_sha256}"
        )

if f0_errors:
    raise RuntimeError(
        "Immutable CSSF source verification failed:\n  - "
        + "\n  - ".join(f0_errors)
    )

FRAMEWORK_IDENTITY = framework_identity_report(PROJECT_ROOT)
if not FRAMEWORK_IDENTITY["frozen_core_valid"]:
    raise RuntimeError("Frozen CSNN-T/GCV core verification failed.")

project_tree_digest = hashlib.sha256(
    "\n".join(
        f"{path}:{digest}"
        for path, digest in sorted(f0_manifest["files"].items())
    ).encode("utf-8")
).hexdigest()

INTEGRITY_GATE = {
    "status": "pass",
    "project_tree_sha256": project_tree_digest,
    "notebook_source_sha256": sha256_file(NOTEBOOK_PATH),
    "immutable_source_files": 67,
    "framework_identity": FRAMEWORK_IDENTITY,
}

print({
    "framework_identity": "PASS",
    "immutable_source_files": INTEGRITY_GATE["immutable_source_files"],
    "case300_shape": FRAMEWORK_IDENTITY["case300_shape"],
    "case300_modeA_features": FRAMEWORK_IDENTITY["case300_modeA_features"],
    "project_tree_sha256": INTEGRITY_GATE["project_tree_sha256"],
})

In [ ]:
# 09 — real Colab runtime gates: CUDA GPU only, with no CPU/QPU fallback
import platform

import numpy as np
import torch
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError('NVIDIA CUDA is unavailable; CPU fallback is prohibited.')

tn_probe = AerSimulator(method='tensor_network', device='GPU', precision='double')
if 'GPU' not in set(tn_probe.available_devices()):
    raise RuntimeError('The Qiskit Aer GPU device is unavailable.')
if 'tensor_network' not in set(tn_probe.available_methods()):
    raise RuntimeError('The Qiskit Aer tensor_network method is unavailable.')
probe = QuantumCircuit(2, 2)
probe.h(0)
probe.cx(0, 1)
probe.measure([0, 1], [0, 1])
probe_result = tn_probe.run(
    transpile(probe, tn_probe), shots=256, seed_simulator=11
).result()
if not probe_result.success:
    raise RuntimeError('Qiskit Aer GPU tensor_network probe failed.')

double_probe = torch.tensor([1.0, 2.0], device='cuda', dtype=torch.float64)
if double_probe.dtype != torch.float64 or not torch.isfinite(double_probe).all():
    raise RuntimeError('CUDA double precision probe failed.')

ARCHITECTURE = 'cssf_qa_dwave_evidence_v60'

runtime_environment = {
    'python': platform.python_version(),
    'cuda_device': torch.cuda.get_device_name(0),
    'cuda_capability': list(torch.cuda.get_device_capability(0)),
    'torch_cuda': torch.version.cuda,
    'architecture': ARCHITECTURE,
    'cpu_fallback': False,
    'qpu_used': False,
}
print(runtime_environment)

In [ ]:
# 09A — numerical stability, Pegasus topology, and solver-identity gate
from core.gcv_stable_v53 import tikhonov_solve_svd_v53, spectral_condition_audit_v53
from dwave_backend.pegasus_fabric_v53 import (
    programmable_pegasus_fabric_qubit_count,
    validate_pegasus_solver_id_v53,
    pegasus_solver_family_v53,
)

PEGASUS_M = 16
NOMINAL_P16_NODES = 24 * PEGASUS_M * (PEGASUS_M - 1)
FABRIC_P16_NODES = programmable_pegasus_fabric_qubit_count(PEGASUS_M)
if NOMINAL_P16_NODES != 5760 or FABRIC_P16_NODES != 5640:
    raise RuntimeError('Pegasus P16 topology semantics gate failed.')
if NOMINAL_P16_NODES - FABRIC_P16_NODES != 120:
    raise RuntimeError('Unexpected P16 structurally excluded nominal-node count.')

# Current and replay-compatible Pegasus solver identities are validated explicitly.
for _sid, _family in (
    ('Advantage_system4', 'Advantage_system4'),
    ('Advantage_system6', 'Advantage_system6'),
    ('Advantage_system4.1', 'Advantage_system4'),
    ('Advantage_system6.4', 'Advantage_system6'),
):
    if validate_pegasus_solver_id_v53(_sid) != _sid:
        raise RuntimeError(f'Current/historical Pegasus solver-id gate failed: {_sid}')
    if pegasus_solver_family_v53(_sid) != _family:
        raise RuntimeError(f'Pegasus solver-family gate failed: {_sid}')
for _bad in ('Advantage2_system4', 'Advantage_system60', 'Advantage_system6_beta'):
    try:
        validate_pegasus_solver_id_v53(_bad)
    except Exception:
        pass
    else:
        raise RuntimeError(f'Forbidden non-Pegasus/lookalike solver id accepted: {_bad}')

print({
    'numerical_solver':'svd_tikhonov_no_normal_equations',
    'pegasus_m':PEGASUS_M,
    'full_nominal_nodes':NOMINAL_P16_NODES,
    'programmable_fabric_nodes':FABRIC_P16_NODES,
    'structurally_excluded_nominal_nodes':NOMINAL_P16_NODES-FABRIC_P16_NODES,
    'real_qpu_topology_policy':'use exact solver working graph + graph_id; never synthesize missing qubits',
    'current_solver_ids':['Advantage_system4','Advantage_system6'],
    'historical_solver_ids_supported_for_replay':['Advantage_system4.1','Advantage_system6.4'],
})

# Runtime API consistency gate for candidate-ranked embedding reuse.
import inspect as _inspect
from experiments_dwave.factorial_runtime_v54 import build_arm_runtime as _build_arm_runtime_api, remap_embedding_by_candidate_rank as _remap_embedding_api
from experiments_dwave.integrated_bess_v54 import build_bess_arm_problem as _build_bess_arm_problem_api
from qubo.model import QUBOModel as _QUBOModel_api

_runtime_sig = _inspect.signature(_build_arm_runtime_api)
if 'frozen_embedding' not in _runtime_sig.parameters:
    raise RuntimeError('build_arm_runtime must accept frozen_embedding= used by cell 42.')
if not hasattr(_QUBOModel_api, 'variable_order'):
    # dataclass slot fields are visible through __dataclass_fields__ even when
    # not exposed as a class-level value; this branch is only a defensive check.
    if 'variable_order' not in getattr(_QUBOModel_api, '__dataclass_fields__', {}):
        raise RuntimeError('QUBOModel variable_order API is unavailable.')
if '.model.variables' in _inspect.getsource(_remap_embedding_api):
    raise RuntimeError('Candidate remapping must use the canonical variable-order API.')

print({'runtime_integration_gate':'PASS','frozen_embedding_keyword':True,'qubo_variable_api':'variable_order','trig_arm_fit':'stable_svd_tikhonov'})

## Evidence program

The program separates five questions: (1) whether CSSF recovers the application-domain periodic structure; (2) whether the planned annealing-control experiment is analytically identifiable before simulation; (3) whether GPU SQA exhibits the predicted structured response; (4) whether CSSF-guided controls improve matched quantum-annealing outcomes; and (5) whether those outcomes survive independent physical/application validation across case300, IEEE-33, Few-FEM motor, and EEG tasks.

## Experiment 1 — Canonical CSNN-T/GCV application-domain evidence

In [ ]:
# 10 — canonical frozen CSNN-T/GCV Level-I evidence
from experiments_dwave.evidence_program_v53 import run_level1_csnn_t_evidence
LEVEL_I_RESULT = run_level1_csnn_t_evidence(PROJECT_ROOT, bootstrap_samples=4000, seed=20260817)
print(json.dumps(LEVEL_I_RESULT, indent=2))
if LEVEL_I_RESULT.get('cssf_numerical_solver') != 'svd_tikhonov_v53':
    raise RuntimeError('v53 Level-I must use the stable SVD Tikhonov realization.')

print({'conditioning': LEVEL_I_RESULT.get('cssf_conditioning', {}), 'gcv_boundary_hit': LEVEL_I_RESULT.get('cssf_conditioning', {}).get('gcv_boundary_hit')})

# Block III — Integrated CSSF(QA) evidence chain

The integrated chain is: periodic application physics → CSSF representation → BESS/QUBO → calibration-resolved annealing controls → Analytical Preflight → GPU SQA response → CSSF response model → independently verified physical endpoint. Each transition is validated separately so that representation effects, control effects, annealer effects, and application value are not conflated.

In [ ]:
# 41 — evidence-program imports and result roots
from dataclasses import asdict
from pathlib import Path
import hashlib, json
import numpy as np

from benchmarks import reference_competitors as rc
from core.types import SurrogateLevel
from experiments_dwave.application_endpoint_v38 import run_application_endpoint, evaluate_confirmation_buses
from experiments_dwave.benchmark_protocol import MatchedControlProtocol, TARGET_NAMES
from experiments_dwave.analytical_preflight import run_analytical_preflight, require_experiment_preflight
from experiments_dwave.claim_set_v38 import PRIMARY_EXTERNAL_COMPARATORS
from experiments_dwave.competitor_fidelity import run_reference_reproduction_suite
from experiments_dwave.contract_evidence_v38 import (
    application_validation_events, campaign_payload_to_trace, confirmation_values, paired_bootstrap_row, placement_resource_events, rename_trace,
    write_competitor_fidelity, write_cost_to_target, write_external_claim_set,
    write_external_export_stub, write_factorial_evidence, write_highs_reference,
    write_operator_action_evidence, write_partitions, write_reproducibility,
    write_resource_accounting, write_statistics, write_json,
)
from experiments_dwave.csnnt_audit_v38 import write_csnnt_fit_audit
from experiments_dwave.evidence_v38 import EvidenceRecorder, canonical_json_hash, sha256_file
from experiments_dwave.experiment_tasks import advance_matched_campaign, qzero_claim_gate, run_strong_classical_benchmark
from experiments_dwave.factorial_campaign_v38 import advance_factorial_campaign, load_factorial_campaign, campaign_to_trace
from experiments_dwave.factorial_runtime_v54 import (
    build_arm_runtime, remap_embedding_by_candidate_rank, select_placement_from_response,
    select_placement_from_sampleset, select_production_placement, select_production_placement_for_control, select_production_placement_for_schedule,
)
from experiments_dwave.hierarchical_refinement_v38 import rank_observed_controls_with_full_hierarchy
from experiments_dwave.integrated_bess_v54 import build_bess_arm_problem, factorial_specs
from experiments_dwave.operator_phase import APPROVED_FAMILIES, load_calibration
from experiments_dwave.qzero_runtime_v38 import run_qzero_matched_full
from experiments_dwave.residual_program_v53 import fit_residual_hierarchy
from experiments_dwave.residual_teacher_v38 import collect_residual_reference_stack
from experiments_dwave.verifiers_v53 import VerificationContext, run_all, PASS
from experiments_dwave.director_matrix_v53 import (
    DEFAULT_DIRECTOR_PLAN, advance_raw_trig_corpus_v47_step, analyze_raw_trig_corpus_v47,
    advance_director_corpus_step, run_d45_01_segmented_vs_global, run_d45_02_dictionary_gate,
    run_d45_03_multioutput_vs_scalar, advance_d45_04_incumbent_step, run_d45_05_query_frontier_gate,
    run_d45_06_control_leverage, run_d45_07_filter_sensitivity, advance_d45_08_target_step,
    run_d45_08_transfer_analysis, advance_d45_10_physical_mass_step, run_d45_09_physical_cost_to_target,
    director_matrix_gate, FOUR_TASK_EXPERIMENTS, IEEE33_EXPERIMENTS, MOTOR_EXPERIMENTS, EEG_EXPERIMENTS,
    validate_four_task_registry, write_four_task_manifest, compare_wrap_resilience, pairwise_binary_features,
    validate_qubo_bridge, fit_pairwise_qubo_bridge, periodic_signal_audit, few_fem_cost_to_target,
    ApplicationResultSchema, validate_application_result_payload, application_fail_closed_gate,
    ieee33_result_schemas, motor_result_schemas, conditional_result_placeholders, four_task_gate,
)
from experiments_dwave.eeg_phase_syntax_v51 import (
    EEGResultSchema, aligned_lagged_features, assert_prediction_alignment,
    phase_periodicity_audit, markov_transition_matrix, first_order_markov_surrogates,
    pooled_training_nll, heldout_partition_nll, pairwise_merge_cost_matrix,
    pairwise_partition_energy, partition_bridge_fidelity, qubo_assignment_graph_stats,
    gcv_real_target_diagnostics, weak_coupling_exponent_audit, eeg_fail_closed_gate,
)

EVIDENCE_ROOT = PROJECT_ROOT / 'results' / 'dwave_evidence_v60' / 'simulator'
CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints' / 'dwave_evidence_v60' / 'simulator'
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
FACTORIAL_ROOT = CHECKPOINT_ROOT / 'D0_D3'
EXTERNAL_ROOT = CHECKPOINT_ROOT / 'external_competitors'
FACTORIAL_ROOT.mkdir(parents=True, exist_ok=True); EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)
PROTOCOL = MatchedControlProtocol()
RECORDER = EvidenceRecorder(EVIDENCE_ROOT, run_id='CSSF-QA-SIMULATOR-v56', mode='simulator')
write_external_claim_set(EVIDENCE_ROOT)
write_external_export_stub(EVIDENCE_ROOT, claim_grade=False)

DIRECTOR_ROOT = EVIDENCE_ROOT / 'evidence_matrix'
DIRECTOR_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
# 42 — D0/D1/D2/D3 application and QUBO arms (CPU only)
RAW_ARM = build_bess_arm_problem(PROJECT_ROOT, 'raw')
TRIG_ARM = build_bess_arm_problem(PROJECT_ROOT, 'trig')
if TRIG_ARM.domain.diagnostics.get('numerical_realization') != 'svd_tikhonov_v53':
    raise RuntimeError('TRIG BESS arm must use the stable SVD-Tikhonov numerical realization.')
if tuple(TRIG_ARM.problem.variable_order) != tuple(TRIG_ARM.problem.model.variable_order):
    raise RuntimeError('TRIG BESS encoding/model variable order mismatch.')
if tuple(RAW_ARM.problem.variable_order) != tuple(RAW_ARM.problem.model.variable_order):
    raise RuntimeError('RAW BESS encoding/model variable order mismatch.')
print({
    'raw_problem': RAW_ARM.problem.fingerprint(),
    'trig_problem': TRIG_ARM.problem.fingerprint(),
    'raw_candidates': len(RAW_ARM.problem.variable_order),
    'trig_candidates': len(TRIG_ARM.problem.variable_order),
})

## Mandatory Analytical Preflight — zero GPU/QPU calls

Before the Pegasus simulator is constructed or sampled, one CPU-only pass evaluates every planned SQA experiment. It certifies admissible forward schedules, first-order CSSF identifiability, conditioning, protected harmonic-dictionary rank, D0–D3 causal contrast completeness, QUBO finiteness/cardinality, and read-budget resolution. The forecast is registered before SQA and cannot be rewritten after observing GPU outcomes.

In [ ]:
# 43 — analytical forecast and experiment-validity certificate; annealer_calls == 0
ANALYTICAL_PREFLIGHT = run_analytical_preflight(
    PROJECT_ROOT,
    protocol=PROTOCOL,
    raw_arm=RAW_ARM,
    trig_arm=TRIG_ARM,
    calibration_family='Advantage_system6',
    comparison_family='Advantage_system4',
    output_path=EVIDENCE_ROOT / 'analytical_preflight.json',
)
print(json.dumps(ANALYTICAL_PREFLIGHT.summary(), indent=2))
print(json.dumps({'prospective_execution_plan':ANALYTICAL_PREFLIGHT.payload['execution_plan'],'application_programs':ANALYTICAL_PREFLIGHT.payload['application_preflight']['programs']}, indent=2))
if not ANALYTICAL_PREFLIGHT.passed:
    raise RuntimeError('Analytical Preflight failed. GPU SQA execution is blocked; inspect analytical_preflight.json.')

In [ ]:
# 44 — Pegasus-P16 simulator runtimes; identical physical chain layout by candidate rank
D3_RUNTIME = build_arm_runtime(PROJECT_ROOT, TRIG_ARM, mode='simulator', calibration_family='Advantage_system6')
D1_RUNTIME = D3_RUNTIME
D3_EMBEDDING = D3_RUNTIME.evaluator.backend.embedding
RAW_EMBEDDING = remap_embedding_by_candidate_rank(TRIG_ARM, RAW_ARM, D3_EMBEDDING)
D0_RUNTIME = build_arm_runtime(
    PROJECT_ROOT, RAW_ARM, mode='simulator', calibration_family='Advantage_system6',
    frozen_embedding=RAW_EMBEDDING,
    chain_strength=D3_RUNTIME.evaluator.backend.chain_strength,
)
D2_RUNTIME = D0_RUNTIME
RUNTIMES = {'D0':D0_RUNTIME,'D1':D1_RUNTIME,'D2':D2_RUNTIME,'D3':D3_RUNTIME}
SPECS = {s.arm_id:s for s in factorial_specs(PROTOCOL)}

NODE_DOMAIN = RECORDER.record_node(
    stage='domain_spectral', output_ids=['domain:trig'],
    metadata={'call_path':TRIG_ARM.domain.diagnostics.get('call_path'), 'model_fingerprint':TRIG_ARM.domain.model_fingerprint},
)
NODE_QUBO = RECORDER.record_node(
    stage='bess_qubo', parents=[NODE_DOMAIN], input_ids=['domain:trig'], output_ids=['bess_qubo:D3'],
    metadata={'problem_fingerprint':TRIG_ARM.problem.fingerprint(), 'candidate_count':len(TRIG_ARM.selection.candidate_buses), 'bess_units':TRIG_ARM.fleet.units_to_place},
)
print({'shared_chain_strength':D3_RUNTIME.evaluator.backend.chain_strength, 'embedding_chains':len(D3_EMBEDDING)})

In [ ]:
# 43 — D0: raw domain + raw control; checkpointed matched campaign
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'D0')
STEP_D0 = advance_factorial_campaign(
    RUNTIMES['D0'].evaluator, arm_id='D0', domain_representation='raw', control_representation='raw',
    protocol=PROTOCOL, project_root=PROJECT_ROOT, campaign_root=FACTORIAL_ROOT,
    backend_identity=RUNTIMES['D0'].backend_manifest, problem_fingerprint=RUNTIMES['D0'].arm.problem.fingerprint(),
)
print(STEP_D0.get('summary', STEP_D0))
if not STEP_D0['complete']:
    raise RuntimeError('D0 checkpoint saved. Re-run this cell to execute the next provenance-locked campaign step; no scientific reduction is permitted.')

In [ ]:
# 44 — D1: trig domain + raw control; checkpointed matched campaign
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'D1')
STEP_D1 = advance_factorial_campaign(
    RUNTIMES['D1'].evaluator, arm_id='D1', domain_representation='trig', control_representation='raw',
    protocol=PROTOCOL, project_root=PROJECT_ROOT, campaign_root=FACTORIAL_ROOT,
    backend_identity=RUNTIMES['D1'].backend_manifest, problem_fingerprint=RUNTIMES['D1'].arm.problem.fingerprint(),
)
print(STEP_D1.get('summary', STEP_D1))
if not STEP_D1['complete']:
    raise RuntimeError('D1 checkpoint saved. Re-run this cell to execute the next provenance-locked campaign step; no scientific reduction is permitted.')

In [ ]:
# 45 — D2: raw domain + operator_phase control; checkpointed matched campaign
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'D2')
STEP_D2 = advance_factorial_campaign(
    RUNTIMES['D2'].evaluator, arm_id='D2', domain_representation='raw', control_representation='operator_phase',
    protocol=PROTOCOL, project_root=PROJECT_ROOT, campaign_root=FACTORIAL_ROOT,
    backend_identity=RUNTIMES['D2'].backend_manifest, problem_fingerprint=RUNTIMES['D2'].arm.problem.fingerprint(),
)
print(STEP_D2.get('summary', STEP_D2))
if not STEP_D2['complete']:
    raise RuntimeError('D2 checkpoint saved. Re-run this cell to execute the next provenance-locked campaign step; no scientific reduction is permitted.')

In [ ]:
# 46 — D3: trig domain + operator_phase control; checkpointed matched campaign
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'D3')
STEP_D3 = advance_factorial_campaign(
    RUNTIMES['D3'].evaluator, arm_id='D3', domain_representation='trig', control_representation='operator_phase',
    protocol=PROTOCOL, project_root=PROJECT_ROOT, campaign_root=FACTORIAL_ROOT,
    backend_identity=RUNTIMES['D3'].backend_manifest, problem_fingerprint=RUNTIMES['D3'].arm.problem.fingerprint(),
)
print(STEP_D3.get('summary', STEP_D3))
if not STEP_D3['complete']:
    raise RuntimeError('D3 checkpoint saved. Re-run this cell to execute the next provenance-locked campaign step; no scientific reduction is permitted.')

In [ ]:
# 47 — exact D0–D3 traces plus atomically collected same-corpus CSSF-raw/no-trig causal ablation
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'raw-trig-shared-corpus')
D0_TRACE = campaign_to_trace(load_factorial_campaign(FACTORIAL_ROOT,'D0'))
D1_TRACE = campaign_to_trace(load_factorial_campaign(FACTORIAL_ROOT,'D1'))
D2_TRACE = campaign_to_trace(load_factorial_campaign(FACTORIAL_ROOT,'D2'))
D3_TRACE = campaign_to_trace(load_factorial_campaign(FACTORIAL_ROOT,'D3'))
RAW_TRIG_CORPUS_PATH = DIRECTOR_ROOT / 'v05_shared_raw_trig_corpus.json'
RAW_TRIG_STEP = advance_raw_trig_corpus_v47_step(
    D3_RUNTIME.evaluator, protocol=PROTOCOL, output_path=RAW_TRIG_CORPUS_PATH, corpus_size=74,
)
print(RAW_TRIG_STEP)
if not RAW_TRIG_STEP['complete']:
    raise RuntimeError('V05 shared corpus checkpoint saved. Re-run this cell for the next single annealer evaluation.')
RAW_TRIG_EVIDENCE = analyze_raw_trig_corpus_v47(
    evaluator=D3_RUNTIME.evaluator, protocol=PROTOCOL, project_root=PROJECT_ROOT,
    corpus_path=RAW_TRIG_CORPUS_PATH, output_path=EVIDENCE_ROOT/'raw_trig_ablation.json', candidate_pool_size=4096,
)
print({'D0':len(D0_TRACE.controls),'D1':len(D1_TRACE.controls),'D2':len(D2_TRACE.controls),'D3':len(D3_TRACE.controls),'shared_corpus_queries':len(RAW_TRIG_EVIDENCE['CSSF-trig']['observation_ids'])})

In [ ]:
# 48 — independent V02 audit of the actual D3 QA-response CSNN-T/GCV fit
from experiments_dwave.cssf_control_v53 import fit_cssf_qa_response
D3_PAYLOAD = load_factorial_campaign(FACTORIAL_ROOT,'D3'); D3_ROWS = D3_PAYLOAD['records']
D3_THETA = np.vstack([np.asarray(r['response']['operator_action'],float) for r in D3_ROWS])
D3_Y = np.asarray([[float(r['response'][k]) for k in TARGET_NAMES] for r in D3_ROWS],float)
nt,nc=PROTOCOL.cssf_initial_train,PROTOCOL.cssf_initial_calibration
D3_CAL=np.arange(nt,nt+nc); D3_ACTIVE=np.arange(nt+nc,len(D3_ROWS)); D3_TRAIN=np.concatenate([np.arange(nt),D3_ACTIVE])
D3_MODEL = fit_cssf_qa_response(D3_THETA[D3_TRAIN],D3_Y[D3_TRAIN],calibration_operator_phase=D3_THETA[D3_CAL],calibration_targets=D3_Y[D3_CAL],target_names=TARGET_NAMES,project_root=PROJECT_ROOT,support_mode='signed_axes',support_order=1)
D3_PHI = D3_MODEL._features(D3_THETA[D3_TRAIN])
write_csnnt_fit_audit(EVIDENCE_ROOT,evidence_id='qa_response:D3',model=D3_MODEL.model,training_features=D3_PHI,training_targets=D3_Y[D3_TRAIN],training_partition_ids=[f'train:d3:{i:04d}' for i in D3_TRAIN],call_path='core.csnn_t_adapter_v53.fit_csnn_t_surrogate')
NODE_PHASE = RECORDER.record_node(stage='operator_phase',parents=[NODE_QUBO],input_ids=['bess_qubo:D3'],output_ids=['operator_phase:D3'],metadata={'calibration_family':'Advantage_system6'})
NODE_SURROGATE = RECORDER.record_node(stage='qa_response_surrogate',parents=[NODE_PHASE],input_ids=['operator_phase:D3'],output_ids=['csnnt:D3'],metadata={'lambda_gcv':float(D3_MODEL.model.lam_opt)})

In [ ]:
# 49 — both frozen calibration families plus schedule-sensitive CUDA SQA proof
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'operator-action-family-check')
sample_controls=np.asarray(D3_TRACE.controls[:2],float)
write_operator_action_evidence(PROJECT_ROOT,EVIDENCE_ROOT,{'Advantage_system4':sample_controls,'Advantage_system6':sample_controls},order=PROTOCOL.order,n_segments=8)
curve6=D3_RUNTIME.evaluator.backend.calibration
sensitivity=[]
for i,c in enumerate(sample_controls):
    t,s=rc.fourier_forward_schedule(c,order=PROTOCOL.order,grid_points=129,reject_nonmonotone=True)
    drive=rc.calibrated_sqa_drive_schedule(t,s,{'s':curve6.s,'A_GHz':curve6.A_GHz,'B_GHz':curve6.B_GHz},anneal_steps=(D3_RUNTIME.evaluator.backend.simulator_sweeps-D3_RUNTIME.evaluator.backend.simulator_burn_in_sweeps),beta_range=(0.1,5.0))
    r=D3_RUNTIME.evaluator(c,num_reads=512,sampling_seed=20260817+i)
    sensitivity.append({'control_id':f'sensitivity:{i}','family':'Advantage_system6','t_us':t.tolist(),'s':s.tolist(),'anneal_steps':(D3_RUNTIME.evaluator.backend.simulator_sweeps-D3_RUNTIME.evaluator.backend.simulator_burn_in_sweeps),'beta_range':[0.1,5.0],'beta_eff':np.asarray(drive['beta_eff']).tolist(),'field_eff':np.asarray(drive['field_eff']).tolist(),'outcome_fingerprint':canonical_json_hash({k:r[k] for k in ('mean_energy','elite_probability','feasibility_probability')},prefix='CSSF-SQA-outcome-v38')})
write_json(EVIDENCE_ROOT/'simulator_schedule_sensitivity.json',{'schema':'CSSF-SIMULATOR-SENSITIVITY-v38','device':'cuda','classical_fallback':False,'schedules':sensitivity})

In [ ]:
# 50 — full OPF→QAOA→MA-QAOA→digitized-QA residual hierarchy; every component enters the final CSSF predictor
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'residual-hierarchy')
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'CSSF-full')
residual_indices=np.linspace(0,len(D3_TRACE.controls)-1,16,dtype=int)
residual_controls=[D3_TRACE.controls[i] for i in residual_indices]
def _residual_key(c): return canonical_json_hash(np.asarray(c,float).round(12).tolist(),prefix='CSSF-v38-residual-control-key')
cached_digitized={_residual_key(D3_TRACE.controls[i]):D3_TRACE.responses[i] for i in residual_indices}
features_by_level,references_by_level,RESIDUAL_TEACHERS = collect_residual_reference_stack(
    residual_controls,evaluator=D3_RUNTIME.evaluator,calibration=D3_RUNTIME.evaluator.backend.calibration,arm=TRIG_ARM,
    ideal_energy=float(D3_RUNTIME.highs.combined_qubo_energy),mode='simulator',seed=20260817,shots=4096,cached_digitized=cached_digitized,
)
RESIDUAL_HIERARCHY = fit_residual_hierarchy(PROJECT_ROOT,features_by_level,references_by_level,target_names=('mean_energy','feasibility_probability'),mode='simulator',sample_ids=[f'residual:{i:04d}' for i in range(len(residual_controls))],source_evidence_ids={SurrogateLevel.OPF:'domain:D3',SurrogateLevel.QAOA:'teacher:QAOA',SurrogateLevel.MA_QAOA:'teacher:MA-QAOA',SurrogateLevel.DIGITIZED_QA:'annealer:D3'})
write_json(EVIDENCE_ROOT/'residual_teachers.json',RESIDUAL_TEACHERS); write_json(EVIDENCE_ROOT/'residual_hierarchy.json',RESIDUAL_HIERARCHY.evidence)
HIERARCHICAL_PROPOSAL = rank_observed_controls_with_full_hierarchy(RESIDUAL_HIERARCHY,D3_RUNTIME.evaluator,np.asarray(D3_TRACE.controls,float))
CSSF_FULL_TRACE = rename_trace(D3_TRACE,'CSSF-full')
print(HIERARCHICAL_PROPOSAL.diagnostics)

In [ ]:
# 51 — independent production sampling; validation AC selects each portfolio, never OOD/N-1
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'production-sampling')
PLACEMENTS={}; PLACEMENT_META={}
for aid,tr in [('D0',D0_TRACE),('D1',D1_TRACE),('D2',D2_TRACE),('D3',D3_TRACE)]:
    PLACEMENTS[aid],PLACEMENT_META[aid]=select_production_placement(PROJECT_ROOT,RUNTIMES[aid],tr,production_reads=8192,replicates=4,seed=20260817)
PLACEMENTS['CSSF-full'],PLACEMENT_META['CSSF-full']=select_production_placement_for_control(PROJECT_ROOT,D3_RUNTIME,HIERARCHICAL_PROPOSAL.control,method='CSSF-full',production_reads=8192,replicates=4,seed=20260817)
HIGHS_PLACEMENT = TRIG_ARM.problem.decode(np.asarray(D3_RUNTIME.highs.selected_sample,dtype=np.int8)); PLACEMENTS['HiGHS']=HIGHS_PLACEMENT
write_highs_reference(EVIDENCE_ROOT,highs=D3_RUNTIME.highs,problem=TRIG_ARM.problem)
NODE_ANNEAL = RECORDER.record_node(stage='annealer',parents=[NODE_SURROGATE],input_ids=['csnnt:D3'],output_ids=['annealer:D3'],metadata={'backend':'Pegasus-P16 CUDA SQA','selected_control':HIERARCHICAL_PROPOSAL.control.tolist()})
NODE_PLACE = RECORDER.record_node(stage='placement',parents=[NODE_ANNEAL],input_ids=['annealer:D3'],output_ids=['placement:CSSF-full'],metadata={'selected_buses':list(PLACEMENTS['CSSF-full'].selected_buses)})

In [ ]:
# 52 — GP+EI-full; one checkpointed matched-campaign advance per invocation
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'GP+EI-full')
STEP = advance_matched_campaign(
    'GP+EI-full', D3_RUNTIME.evaluator, PROTOCOL, project_root=PROJECT_ROOT,
    campaign_root=EXTERNAL_ROOT, backend_identity=D3_RUNTIME.backend_manifest,
    problem_fingerprint=TRIG_ARM.problem.fingerprint(),
)
print(STEP)
if not STEP['complete']:
    raise RuntimeError('GP+EI-full checkpoint saved. Re-run this cell for the next provenance-locked campaign step.')
GPEI_PAYLOAD = json.loads((EXTERNAL_ROOT/'GP+EI-full.json').read_text())
GPEI_TRACE = campaign_payload_to_trace(GPEI_PAYLOAD)
print(GPEI_TRACE.method, len(GPEI_TRACE.controls))

In [ ]:
# 53 — Finzgar-BO-matched-full; one checkpointed matched-campaign advance per invocation
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'Finzgar-BO-matched-full')
STEP = advance_matched_campaign(
    'Finzgar-BO-matched-full', D3_RUNTIME.evaluator, PROTOCOL, project_root=PROJECT_ROOT,
    campaign_root=EXTERNAL_ROOT, backend_identity=D3_RUNTIME.backend_manifest,
    problem_fingerprint=TRIG_ARM.problem.fingerprint(),
)
print(STEP)
if not STEP['complete']:
    raise RuntimeError('Finzgar-BO-matched-full checkpoint saved. Re-run this cell for the next provenance-locked campaign step.')
FINZGAR_PAYLOAD = json.loads((EXTERNAL_ROOT/'Finzgar-BO-matched-full.json').read_text())
FINZGAR_TRACE = campaign_payload_to_trace(FINZGAR_PAYLOAD)
print(FINZGAR_TRACE.method, len(FINZGAR_TRACE.controls))

In [ ]:
# 54 — TuRBO-matched-full; one checkpointed matched-campaign advance per invocation
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'TuRBO-matched-full')
STEP = advance_matched_campaign(
    'TuRBO-matched-full', D3_RUNTIME.evaluator, PROTOCOL, project_root=PROJECT_ROOT,
    campaign_root=EXTERNAL_ROOT, backend_identity=D3_RUNTIME.backend_manifest,
    problem_fingerprint=TRIG_ARM.problem.fingerprint(),
)
print(STEP)
if not STEP['complete']:
    raise RuntimeError('TuRBO-matched-full checkpoint saved. Re-run this cell for the next provenance-locked campaign step.')
TURBO_PAYLOAD = json.loads((EXTERNAL_ROOT/'TuRBO-matched-full.json').read_text())
TURBO_TRACE = campaign_payload_to_trace(TURBO_PAYLOAD)
print(TURBO_TRACE.method, len(TURBO_TRACE.controls))

In [ ]:
# 55 — full QZero NN+MCTS pretraining/fine-tuning; no proxy corpus is accepted
QZERO_PREFLIGHT = ANALYTICAL_PREFLIGHT.payload['experiments']['QZero-matched-full']
QZERO_CORPUS = PROJECT_ROOT/'data'/'qzero_full_pretraining_corpus_v38.json'
QZERO_GATE_RAW = qzero_claim_gate(QZERO_CORPUS)
QZERO_RESULT=None; QZERO_TRACE=None
if QZERO_PREFLIGHT['pass'] and QZERO_GATE_RAW['pass']:
    initial_elite=np.asarray([r['elite_probability'] for r in D3_TRACE.responses[:PROTOCOL.cssf_initial_count]],float)
    QZERO_RESULT=run_qzero_matched_full(D3_RUNTIME.evaluator,TRIG_ARM.hamiltonian,protocol=PROTOCOL,corpus_path=QZERO_CORPUS,terminal_success_threshold=float(np.quantile(initial_elite,0.75)))
    QZERO_TRACE=QZERO_RESULT.trace
write_json(EVIDENCE_ROOT/'qzero_gate.json',{'full_qzero_ready':bool(QZERO_GATE_RAW['pass']),'details':QZERO_GATE_RAW,'runtime_executed':QZERO_RESULT is not None})
print(QZERO_GATE_RAW)

In [ ]:
# 56 — full worldline susceptibility and tuned strong SA/Tabu on the identical D3 BQM
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'Worldline-Susceptibility-full')
from experiments_dwave.bess_evidence import to_dimod_bqm
curve=D3_RUNTIME.evaluator.backend.calibration; h=np.asarray(TRIG_ARM.hamiltonian.linear_z,float); J=np.asarray(TRIG_ARM.hamiltonian.quadratic_zz,float); J=J+J.T
WORLDLINE_CONSTRUCTION=rc.construct_worldline_schedule(h,J,s_grid=np.linspace(0,1,21),A_of_s=lambda x:np.interp(x,curve.s,curve.A_GHz),B_of_s=lambda x:np.interp(x,curve.s,curve.B_GHz),T_us=20.0,chi0=1e-6,config=rc.WorldlineSQAConfig(beta=2.0,replicas=32,burn_in_sweeps=200,measurement_sweeps=800,thin=4,seed=20260817))
WORLDLINE_RESPONSE=D3_RUNTIME.evaluator.backend.evaluate_schedule(np.asarray(WORLDLINE_CONSTRUCTION['t_us'],float),np.asarray(WORLDLINE_CONSTRUCTION['schedule_s'],float),num_reads=PROTOCOL.reads_per_control,elite_threshold=D3_RUNTIME.evaluator.elite_threshold,feasibility=D3_RUNTIME.evaluator.feasibility,success_energy=D3_RUNTIME.evaluator.success_energy,label='Worldline-Susceptibility-full')
BQM=to_dimod_bqm(TRIG_ARM.problem)
STRONG_RESULT=run_strong_classical_benchmark(BQM,elite_threshold=D3_RUNTIME.evaluator.elite_threshold,output_path=EVIDENCE_ROOT/'strong_classical.json',num_reads=512)

In [ ]:
# 57 — application placements for every full external comparator
for name,tr in [('GP+EI-full',GPEI_TRACE),('Finzgar-BO-matched-full',FINZGAR_TRACE),('TuRBO-matched-full',TURBO_TRACE)]:
    PLACEMENTS[name],PLACEMENT_META[name]=select_production_placement(PROJECT_ROOT,D3_RUNTIME,tr,production_reads=8192,replicates=4,seed=20260817)
if QZERO_TRACE is not None:
    PLACEMENTS['QZero-matched-full'],PLACEMENT_META['QZero-matched-full']=select_production_placement(PROJECT_ROOT,D3_RUNTIME,QZERO_TRACE,production_reads=8192,replicates=4,seed=20260817)
PLACEMENTS['Worldline-Susceptibility-full'],PLACEMENT_META['Worldline-Susceptibility-full']=select_production_placement_for_schedule(
    PROJECT_ROOT,D3_RUNTIME,np.asarray(WORLDLINE_CONSTRUCTION['t_us'],float),np.asarray(WORLDLINE_CONSTRUCTION['schedule_s'],float),
    method='Worldline-Susceptibility-full',production_reads=8192,replicates=4,seed=20260817,
)
from dwave.samplers import SimulatedAnnealingSampler, TabuSampler
SA_SS=SimulatedAnnealingSampler().sample(BQM,num_reads=2048,seed=20260901,**dict(STRONG_RESULT['Strong-SA']['tuning']['best_config']))
TABU_SS=TabuSampler().sample(BQM,num_reads=2048,seed=20260902,**dict(STRONG_RESULT['Strong-Tabu']['tuning']['best_config']))
PLACEMENTS['Strong-SA'],PLACEMENT_META['Strong-SA']=select_placement_from_sampleset(PROJECT_ROOT,TRIG_ARM,SA_SS,method='Strong-SA')
PLACEMENTS['Strong-Tabu'],PLACEMENT_META['Strong-Tabu']=select_placement_from_sampleset(PROJECT_ROOT,TRIG_ARM,TABU_SS,method='Strong-Tabu')
missing=set(PRIMARY_EXTERNAL_COMPARATORS)-set(PLACEMENTS)
print({'external_placements_complete':not missing,'missing':sorted(missing)})

In [ ]:
# 58 — one identical AC/OOD/full-N-1 endpoint for D0–D3, CSSF-full, HiGHS and all full competitors
APPLICATION_ENDPOINT = run_application_endpoint(PROJECT_ROOT,PLACEMENTS,output_path=EVIDENCE_ROOT/'application_endpoint.json',selection_input_ids=[f'validation:selection:{i:04d}' for i in range(96)])
ENDPOINT_ROWS={row['method']:row for row in APPLICATION_ENDPOINT['placements']}
NODE_ENDPOINT = RECORDER.record_node(stage='application_endpoint',parents=[NODE_PLACE],input_ids=['placement:CSSF-full'],output_ids=['endpoint:CSSF-full'],metadata={'full_n1_count':APPLICATION_ENDPOINT['full_n1_count']})
LINEAGE_ENDPOINT_IDS={'D0':'endpoint:D0','D1':'endpoint:D1','D2':'endpoint:D2','D3':NODE_ENDPOINT}
for aid in ('D0','D1','D2'):
    RECORDER.record_node(stage='application_endpoint',input_ids=[f'placement:{aid}'],output_ids=[f'endpoint:{aid}'],node_id=f'endpoint:{aid}',metadata={'arm_id':aid})
write_factorial_evidence(EVIDENCE_ROOT,specs=list(SPECS.values()),endpoint_rows={aid:ENDPOINT_ROWS[aid] for aid in ('D0','D1','D2','D3')},lineage_endpoint_ids=LINEAGE_ENDPOINT_IDS)

In [ ]:
# 59 — independent application-level confirmation; one method-confirmation task per invocation
CONFIRMATION_CHECKPOINT = DIRECTOR_ROOT / 'application_confirmation_checkpoint.json'
if CONFIRMATION_CHECKPOINT.is_file():
    _CONFIRM_PAYLOAD = json.loads(CONFIRMATION_CHECKPOINT.read_text(encoding='utf-8'))
else:
    _CONFIRM_PAYLOAD = {'schema':'CSSF-QA-CONFIRMATION-CHECKPOINT-v47','confirmations':{}}
CONFIRMATIONS = dict(_CONFIRM_PAYLOAD.get('confirmations', {}))
_CONFIRM_METHODS = ['CSSF-full','HiGHS'] + [name for name in PRIMARY_EXTERNAL_COMPARATORS if name in PLACEMENTS]
_pending = [name for name in _CONFIRM_METHODS if name not in CONFIRMATIONS]
if _pending:
    name = _pending[0]
    CONFIRMATIONS[name] = evaluate_confirmation_buses(PROJECT_ROOT, PLACEMENTS[name].selected_buses, method=name)
    write_json(CONFIRMATION_CHECKPOINT, {'schema':'CSSF-QA-CONFIRMATION-CHECKPOINT-v47','confirmations':CONFIRMATIONS,'complete':len(CONFIRMATIONS)==len(_CONFIRM_METHODS)})
    print({'confirmed':name,'remaining':len(_pending)-1})
    if len(_pending) > 1:
        raise RuntimeError('Independent confirmation checkpoint saved. Re-run this cell for the next method.')
CSSF_CONFIRM = CONFIRMATIONS['CSSF-full']
CSSF_VALUES = confirmation_values(CSSF_CONFIRM); STAT_ROWS=[]
for j,name in enumerate(PRIMARY_EXTERNAL_COMPARATORS):
    if name not in CONFIRMATIONS: continue
    comp = CONFIRMATIONS[name]
    STAT_ROWS.append(paired_bootstrap_row(competitor=name,cssf_values=CSSF_VALUES,competitor_values=confirmation_values(comp),seed=20261000+j,alpha=0.05,margin=0.0,bootstrap_samples=4000))
missing_stats=sorted(set(PRIMARY_EXTERNAL_COMPARATORS)-{r['competitor'] for r in STAT_ROWS})
if missing_stats:
    write_json(EVIDENCE_ROOT/'statistics.json',{'schema':'CSSF-QA-PAIRED-STATISTICS-v38','comparisons':STAT_ROWS,'missing_comparators':missing_stats,'claim_locked':True})
else:
    write_statistics(EVIDENCE_ROOT,STAT_ROWS)
print({'confirmation_rows':len(STAT_ROWS),'missing':missing_stats,'reference_confirmation':'HiGHS' in CONFIRMATIONS})

In [ ]:
# 60 — full resource accounting, cost-to-target and competitor-fidelity product
TRACE_LIST=[rename_trace(D0_TRACE,'D0'),rename_trace(D1_TRACE,'D1'),rename_trace(D2_TRACE,'D2'),rename_trace(D3_TRACE,'D3'),CSSF_FULL_TRACE,GPEI_TRACE,FINZGAR_TRACE,TURBO_TRACE]
if QZERO_TRACE is not None: TRACE_LIST.append(QZERO_TRACE)
EXTRA_EVENTS=placement_resource_events(PLACEMENT_META)+application_validation_events(list(PLACEMENTS),validation=96,ood=192,n1=int(APPLICATION_ENDPOINT['full_n1_count']),confirmation=0)+[{'method':'Worldline-Susceptibility-full','event_type':'evaluation','resource':'control_evaluation','amount':1.0,'metadata':{'utility':float(WORLDLINE_RESPONSE['elite_probability'])}},{'method':'Worldline-Susceptibility-full','event_type':'evaluation','resource':'annealer_reads','amount':float(WORLDLINE_RESPONSE['num_reads']),'metadata':{'utility':float(WORLDLINE_RESPONSE['elite_probability'])}}]
for name in ['CSSF-full',*PRIMARY_EXTERNAL_COMPARATORS]:
    if name in PLACEMENTS: EXTRA_EVENTS.append({'method':name,'event_type':'application_confirmation','resource':'physical_validation_cases','amount':64.0,'metadata':{'partition':'confirmation'}})
for name in ('Strong-SA','Strong-Tabu'):
    EXTRA_EVENTS.append({'method':name,'event_type':'evaluation','resource':'classical_reference','amount':float(STRONG_RESULT[name]['total_classical_seconds']),'metadata':{'utility':float(STRONG_RESULT[name]['holdout_mean'])}})
EVENTS,BUDGET=write_resource_accounting(EVIDENCE_ROOT,traces=TRACE_LIST,extra_events=EXTRA_EVENTS,identical_schedule_methods=['D0','D1','D2','D3','GP+EI-full','Finzgar-BO-matched-full','TuRBO-matched-full'],charged_methods=['CSSF-full','QZero-matched-full','Worldline-Susceptibility-full','Strong-SA','Strong-Tabu'])
initial_target=float(np.quantile([r['elite_probability'] for r in D3_TRACE.responses[:PROTOCOL.cssf_initial_count]],0.90))
write_cost_to_target(EVIDENCE_ROOT,events=EVENTS,methods=[m for m in ('CSSF-full','GP+EI-full','Finzgar-BO-matched-full','TuRBO-matched-full','QZero-matched-full','Worldline-Susceptibility-full') if m in BUDGET['methods']],target=initial_target,target_definition={'partition':'initial_design','quantile':0.90,'target':'elite_probability'})
REPRO_SUITE=run_reference_reproduction_suite(run_heavy_finzgar=True,run_heavy_qzero=True)
write_competitor_fidelity(PROJECT_ROOT,EVIDENCE_ROOT,reproduction_suite=REPRO_SUITE,qzero_gate=QZERO_GATE_RAW,strong_classical_result=STRONG_RESULT)

In [ ]:
# 61 — split/leakage manifest and deterministic reproducibility evidence
parts={'train':[f'train:{i:04d}' for i in range(PROTOCOL.cssf_initial_train)],'calibration':[f'calibration:{i:04d}' for i in range(PROTOCOL.cssf_initial_calibration)],'validation':[f'validation:{i:04d}' for i in range(96)],'ood':[f'ood:{i:04d}' for i in range(192)],'n1':[f'n1:{i:04d}' for i in range(APPLICATION_ENDPOINT['full_n1_count'])],'confirmation':[f'confirmation:{i:04d}' for i in range(64)]}
observations=[]
for partition,ids in parts.items():
    observations.extend({'provenance_id':f'{partition}:prov:{i:04d}','partition':partition} for i,_ in enumerate(ids))
write_partitions(EVIDENCE_ROOT,partitions=parts,observations=observations)
rebuild=build_bess_arm_problem(PROJECT_ROOT,'trig'); deterministic_ok=rebuild.problem.fingerprint()==TRIG_ARM.problem.fingerprint() and rebuild.domain.model_fingerprint==TRIG_ARM.domain.model_fingerprint
write_reproducibility(PROJECT_ROOT,EVIDENCE_ROOT,seeds={'global':20260817,'factorial':PROTOCOL.seed},config_hashes={'base':sha256_file(PROJECT_ROOT/'config'/'base.yaml'),'case300':sha256_file(PROJECT_ROOT/'config'/'case300.yaml'),'emulator_gpu':sha256_file(PROJECT_ROOT/'config'/'emulator_gpu.yaml')},code_paths=['core/csnn_t.py','core/gcv.py','experiments_dwave/integrated_bess_v38.py','experiments_dwave/verifiers_v38/aggregate.py'],deterministic_regeneration_passed=deterministic_ok)
RECORDER.finalize(run_manifest={'program':'CSSF_QA_DWave_Scalable_Evidence_v60','notebook':NOTEBOOK_PATH.name,'framework':'CSSF four-task reproducible evidence program'})

# Block IV — protected evidence matrix: causal mechanism → matched frontier → physical/economic value

These ten closures form the protected evidence matrix: causal mechanism, matched frontier, transfer, control leverage, physical superior-set mass, and cost-to-confirmed-target. Protected partitions are kept separate, and execution completeness is reported independently from whether a positive hypothesis passes.

In [ ]:
# 63 — protected response corpus; one annealer response per invocation
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'director-corpus')
DIRECTOR_CORPUS_PATH = DIRECTOR_ROOT / 'protected_response_corpus.json'
DIRECTOR_CORPUS_STEP = advance_director_corpus_step(
    D3_RUNTIME.evaluator, protocol=PROTOCOL, output_path=DIRECTOR_CORPUS_PATH, plan=DEFAULT_DIRECTOR_PLAN,
)
print(DIRECTOR_CORPUS_STEP)
if not DIRECTOR_CORPUS_STEP['complete']:
    raise RuntimeError('Protected response-corpus checkpoint saved. Re-run this cell for the next single annealer evaluation.')

In [ ]:
# 64 — D45-01: segmented operator action versus global integrated-resource baseline
D45_01 = run_d45_01_segmented_vs_global(
    corpus_path=DIRECTOR_CORPUS_PATH, project_root=PROJECT_ROOT,
    output_path=DIRECTOR_ROOT/'d45_01_segmented_vs_global.json', primary_target='elite_probability',
)
print(json.dumps({'execution_complete':D45_01['execution_complete'],'claim_passed':D45_01['claim_passed'],'confirmation_lcb':D45_01['partitions']['confirmation']['paired_squared_error_improvement']['lcb'],'ood_lcb':D45_01['partitions']['ood']['paired_squared_error_improvement']['lcb']},indent=2))

In [ ]:
# 65 — D45-02: protected harmonic-dictionary order/frequency-semantics gate
D45_02 = run_d45_02_dictionary_gate(
    corpus_path=DIRECTOR_CORPUS_PATH, project_root=PROJECT_ROOT,
    output_path=DIRECTOR_ROOT/'d45_02_dictionary_gate.json', primary_target='elite_probability',
)
print(json.dumps({'chosen_dictionary':D45_02.get('chosen_dictionary'),'dictionary_gate_passed':D45_02.get('dictionary_gate_passed'),'selection_frequency':D45_02.get('dictionary_selection_frequency')},indent=2))

In [ ]:
# 66 — D45-03: multi-output CSNN-T response composition versus matched scalar CSSF
D45_03 = run_d45_03_multioutput_vs_scalar(
    corpus_path=DIRECTOR_CORPUS_PATH, project_root=PROJECT_ROOT,
    output_path=DIRECTOR_ROOT/'d45_03_multi_vs_scalar.json', primary_target='elite_probability', minimum_feasibility=0.0,
)
print(json.dumps({'claim_passed':D45_03['claim_passed'],'selected_target_delta':D45_03['selected_target_delta'],'predictive_gate_passed':D45_03['predictive_gate_passed']},indent=2))

In [ ]:
# 67 — D45-04: native D-Wave incumbent reference; candidate scan then frozen-winner independent confirmation
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'D45-04')
D45_04_STEP = advance_d45_04_incumbent_step(
    D3_RUNTIME.evaluator, protocol=PROTOCOL,
    output_path=DIRECTOR_ROOT/'d45_04_dwave_incumbent.json', confirmation_replicates=16,
)
print(D45_04_STEP)
if not D45_04_STEP.get('complete', False):
    raise RuntimeError('D45-04 checkpoint saved. Re-run this cell for the next single incumbent/confirmation evaluation.')

In [ ]:
# 68 — D45-05: 2026 query-efficiency frontier fidelity gate; no proxy is promoted to a full competitor
OPTIMIZED_QZERO_2026_ASSET = PROJECT_ROOT/'data'/'optimized_qzero_2026_claim_asset.json'
D45_05 = run_d45_05_query_frontier_gate(
    original_qzero_gate=QZERO_GATE_RAW,
    optimized_qzero_asset=OPTIMIZED_QZERO_2026_ASSET if OPTIMIZED_QZERO_2026_ASSET.is_file() else None,
    output_path=DIRECTOR_ROOT/'d45_05_query_frontier.json',
)
print(json.dumps(D45_05,indent=2))

In [ ]:
# 69 — D45-06: control-leverage identification and fail-closed abstention state
D45_06 = run_d45_06_control_leverage(
    corpus_path=DIRECTOR_CORPUS_PATH, output_path=DIRECTOR_ROOT/'d45_06_control_leverage.json',
    target='elite_probability', minimum_practical_delta=0.02,
)
print(json.dumps({'state':D45_06['state'],'deattenuated_variance':D45_06['deattenuated_control_variance'],'range_lcb':D45_06['range_bootstrap_lcb']},indent=2))

In [ ]:
# 70 — D45-07: nominal versus finite-bandwidth proxy sensitivity of operator-action coordinates
D45_07 = run_d45_07_filter_sensitivity(
    corpus_path=DIRECTOR_CORPUS_PATH, project_root=PROJECT_ROOT, calibration_family='Advantage_system6',
    protocol=PROTOCOL, output_path=DIRECTOR_ROOT/'d45_07_filter_sensitivity.json', sample_controls=128,
)
print(json.dumps({'median_relative_distortion':D45_07['median_relative_distortion'],'q95_relative_distortion':D45_07['q95_relative_distortion'],'max_relative_distortion':D45_07['max_relative_distortion']},indent=2))

In [ ]:
# 71 — D45-08 target-context corpus: System4 residual-transfer experiment; one target evaluation per invocation
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'D45-08')
SYSTEM4_RUNTIME = build_arm_runtime(
    PROJECT_ROOT, TRIG_ARM, mode='simulator', calibration_family='Advantage_system4',
    frozen_embedding=D3_EMBEDDING, chain_strength=D3_RUNTIME.evaluator.backend.chain_strength,
)
D45_08_TARGET_PATH = DIRECTOR_ROOT/'d45_08_system4_target_corpus.json'
D45_08_STEP = advance_d45_08_target_step(
    SYSTEM4_RUNTIME.evaluator, source_corpus_path=DIRECTOR_CORPUS_PATH, output_path=D45_08_TARGET_PATH,
)
print(D45_08_STEP)
if not D45_08_STEP['complete']:
    raise RuntimeError('D45-08 target-context checkpoint saved. Re-run this cell for the next single target-context annealer evaluation.')

In [ ]:
# 72 — D45-08 protected residual-transfer learning curve versus scratch
D45_08 = run_d45_08_transfer_analysis(
    source_corpus_path=DIRECTOR_CORPUS_PATH, target_corpus_path=D45_08_TARGET_PATH,
    project_root=PROJECT_ROOT, output_path=DIRECTOR_ROOT/'d45_08_transfer_analysis.json',
    target_name='elite_probability',
)
print(json.dumps({'prefrozen_quality_target':D45_08['prefrozen_quality_target'],'transfer_first_hit':D45_08['transfer_first_hit'],'scratch_first_hit':D45_08['scratch_first_hit'],'claim_passed':D45_08['claim_passed']},indent=2))

In [ ]:
# 73 — D45-10: exact sampled physical superior-set probability bridge; one sampling or one physical-label task per invocation
require_experiment_preflight(ANALYTICAL_PREFLIGHT, 'D45-10')
D45_10_STEP = advance_d45_10_physical_mass_step(
    project_root=PROJECT_ROOT, arm=TRIG_ARM, evaluator=D3_RUNTIME.evaluator,
    control=HIERARCHICAL_PROPOSAL.control, highs_placement=HIGHS_PLACEMENT,
    output_path=DIRECTOR_ROOT/'d45_10_physical_mass.json', gamma=0.0, reads=int(ANALYTICAL_PREFLIGHT.payload['execution_plan']['D45-10']['reads']),
)
print(D45_10_STEP)
if not D45_10_STEP.get('complete', False):
    raise RuntimeError('D45-10 checkpoint saved. Re-run this cell for the next single sampling/physical-label task.')

In [ ]:
# 74 — D45-09: fully charged campaign cost to independently confirmed physical target
METHOD_COSTS = {m:dict(v.get('totals',{})) for m,v in BUDGET.get('methods',{}).items()}
D45_09 = run_d45_09_physical_cost_to_target(
    method_costs=METHOD_COSTS, confirmation_by_method=CONFIRMATIONS,
    reference_method='HiGHS', output_path=DIRECTOR_ROOT/'d45_09_physical_cost_to_target.json',
    gamma=0.0, cssf_method='CSSF-full', primary_cost_axis='control_evaluation',
)
print(json.dumps({'physical_target':D45_09['physical_target'],'strongest_reached_competitor':D45_09['strongest_reached_competitor_on_primary_axis'],'commercial_cost_claim_passed':D45_09['commercial_cost_claim_passed']},indent=2))

In [ ]:
# 75 — evidence-matrix execution/claim separation; no positive result is manufactured
EVIDENCE_MATRIX_GATE = director_matrix_gate(DIRECTOR_ROOT)
print(json.dumps(EVIDENCE_MATRIX_GATE,indent=2))
if not EVIDENCE_MATRIX_GATE['evidence_execution_complete']:
    print('Evidence matrix is incomplete; missing/incomplete D45 artifacts remain fail-closed.')
if not EVIDENCE_MATRIX_GATE['broad_query_efficiency_claim_unlocked']:
    print('Broad 2026 query-efficiency leadership remains explicitly locked; narrowed matched-comparator claims are unaffected.')

# Block V — synchronized four-task / three-domain application program

The retained case300 evidence remains intact. This block exposes the IEEE-33 resilience hierarchy, the Few-FEM outer-rotor BLDC program, and the active EEG phase/syntax microstate program used in the four-task evidence program. No placeholder, synthetic implementation check, missing-data plan, simulator-only record, or unconfirmed QPU observation is treated as application evidence.

In [ ]:
# 76 — four-task registry and dependency graph
FOUR_TASK_ROOT = EVIDENCE_ROOT / 'four_task'
FOUR_TASK_ROOT.mkdir(parents=True, exist_ok=True)
FOUR_TASK_MANIFEST = write_four_task_manifest(
    FOUR_TASK_ROOT / 'experiment_registry.json', FOUR_TASK_EXPERIMENTS
)
print(json.dumps(FOUR_TASK_MANIFEST, indent=2))

## IEEE-33 resilience task — requested BESS questions first, full-resource extension second

Execution order is fixed: (1) loss-oriented CSSF+QA BESS placement evaluated under the Mishra fault/WRAP model; (2) resilience-aware BESS objective if transfer is insufficient; (3) only then the full BESS+MG+TL+MSU+reconfiguration program R33-E0…E11. Continuous AC/SOC/dispatch/routing recourse is never relabeled as QUBO.

In [ ]:
# 77 — IEEE-33 experiment plan, deterministic implementation checks, and fail-closed evidence gate
IEEE33_PLAN = [asdict(e) for e in IEEE33_EXPERIMENTS]
# Explicit canonical IDs: R33-RQ1, R33-RQ2, R33-E0, R33-E1, R33-E2, R33-E3, R33-E4, R33-E5, R33-E6, R33-E7, R33-E8, R33-E9, R33-E10, R33-E11.
IEEE33_ASSETS = {
    'reference_case': PROJECT_ROOT / 'data' / 'ieee33_mishra_reference.json',
    'fault_scenarios': PROJECT_ROOT / 'data' / 'ieee33_mishra_fault_scenarios.json',
    'transport_network': PROJECT_ROOT / 'data' / 'ieee33_transport_network.json',
}
IEEE33_ASSET_STATUS = {k: p.is_file() for k,p in IEEE33_ASSETS.items()}
write_json(FOUR_TASK_ROOT / 'ieee33_plan.json', {
    'schema':'CSSF-QA-IEEE33-v51', 'execution_complete':False,
    'experiments':IEEE33_PLAN, 'asset_status':IEEE33_ASSET_STATUS,
    'sequence':['R33-RQ1','R33-RQ2']+[f'R33-E{i}' for i in range(12)],
    'claim_status':'PENDING_REQUIRED_IEEE33_ASSETS_AND_EXECUTION',
})

# Synthetic numerical implementation checks only: verifies metric directions and QUBO bridge machinery,
# not IEEE-33 application evidence.
rng_r33 = np.random.default_rng(330049)
r33_n = 64
r33_ref = {
    'withstand': rng_r33.normal(0.0, 1.0, r33_n),
    'recover': rng_r33.normal(0.0, 1.0, r33_n),
    'adapt': rng_r33.normal(1.0, 0.2, r33_n),
    'prevent': rng_r33.normal(1.0, 0.2, r33_n),
}
r33_cand = {
    'withstand': r33_ref['withstand'] + 0.25,
    'recover': r33_ref['recover'] + 0.20,
    'adapt': r33_ref['adapt'] - 0.15,
    'prevent': r33_ref['prevent'] - 0.10,
}
IEEE33_WRAP_LOCAL = compare_wrap_resilience(r33_ref, r33_cand, bootstrap_samples=2000, seed=330049)
r33_X = rng_r33.integers(0,2,size=(180,8)).astype(float)
r33_F, _ = pairwise_binary_features(r33_X)
r33_coef = rng_r33.normal(size=r33_F.shape[1])
r33_y = r33_F @ r33_coef
IEEE33_QUBO_LOCAL = fit_pairwise_qubo_bridge(
    r33_X, r33_y, train_indices=np.arange(120), validation_indices=np.arange(120,180), ridge=1e-10, top_k=10
)
write_json(FOUR_TASK_ROOT / 'ieee33_local_implementation_checks.json', {
    'schema':'CSSF-QA-IEEE33-LOCAL-IMPLEMENTATION-v51', 'execution_complete':True,
    'synthetic_only':True, 'application_evidence':False, 'claim_passed':False,
    'wrap_direction_check':IEEE33_WRAP_LOCAL, 'qubo_bridge_check':IEEE33_QUBO_LOCAL['validation'],
})
IEEE33_GATE = application_fail_closed_gate(
    FOUR_TASK_ROOT / 'ieee33', ieee33_result_schemas(), output_path=FOUR_TASK_ROOT / 'ieee33_gate.json'
)
print({
    'ieee33_experiments':len(IEEE33_PLAN), 'assets':IEEE33_ASSET_STATUS,
    'local_wrap_direction_pass':IEEE33_WRAP_LOCAL['all_four_positive'],
    'local_qubo_bridge_pass':IEEE33_QUBO_LOCAL['validation']['passed'],
    'application_claim_status':IEEE33_GATE['claim_status'],
})

## Few-FEM outer-rotor BLDC electromagnetic co-design

M59-01…M59-15 cover physical-period verification, periodic representation causality, matched Few-FEM learning curves, QUBO-fidelity gates, direct electric-machine QA prior art, classical solver frontier, QA trigonometrization, System6/System4 campaigns, independent multiphysics FEM, prototype/dyno confirmation, and separate QA contribution.

In [ ]:
# 78 — Few-FEM motor experiment plan, deterministic implementation checks, and fail-closed evidence gate
MOTOR_PLAN = [asdict(e) for e in MOTOR_EXPERIMENTS]
MOTOR_ASSETS = {
    'baseline_cad_manifest': PROJECT_ROOT / 'data' / 'motor_baseline_cad_manifest.json',
    'design_space': PROJECT_ROOT / 'data' / 'motor_design_space.json',
    'fem_corpus': PROJECT_ROOT / 'data' / 'motor_fem_corpus.npz',
    'bench_protocol': PROJECT_ROOT / 'data' / 'motor_bench_protocol.json',
}
MOTOR_ASSET_STATUS = {k: p.is_file() for k,p in MOTOR_ASSETS.items()}
write_json(FOUR_TASK_ROOT / 'motor_plan.json', {
    'schema':'CSSF-QA-FEW-FEM-MOTOR-v51', 'execution_complete':False,
    'experiments':MOTOR_PLAN, 'asset_status':MOTOR_ASSET_STATUS,
    'primary_qpu_context':'Advantage_system6', 'transfer_qpu_context':'Advantage_system4',
    'claim_status':'PENDING_REQUIRED_CAD_FEM_QPU_AND_PHYSICAL_EXECUTION',
})

# Synthetic numerical implementation checks only: verifies periodic-fit and charged Few-FEM cost logic.
theta_motor_demo = np.linspace(0.0, 2*np.pi, 361)
y_motor_demo = 2.0 + 0.8*np.cos(3*theta_motor_demo) - 0.35*np.sin(5*theta_motor_demo)
MOTOR_PERIOD_LOCAL = periodic_signal_audit(theta_motor_demo, y_motor_demo, harmonics=8)
MOTOR_FEW_FEM_LOCAL = few_fem_cost_to_target([
    {'method':'CSSF-demo','fem_calls':[12,24,36,48],'confirmed_utility':[0.62,0.78,0.91,0.94]},
    {'method':'GP-demo','fem_calls':[12,24,36,48],'confirmed_utility':[0.58,0.70,0.82,0.88]},
], target_utility=0.90)
write_json(FOUR_TASK_ROOT / 'motor_local_implementation_checks.json', {
    'schema':'CSSF-QA-MOTOR-LOCAL-IMPLEMENTATION-v51', 'execution_complete':True,
    'synthetic_only':True, 'application_evidence':False, 'claim_passed':False,
    'periodic_signal_check':MOTOR_PERIOD_LOCAL, 'few_fem_cost_check':MOTOR_FEW_FEM_LOCAL,
})
MOTOR_GATE = application_fail_closed_gate(
    FOUR_TASK_ROOT / 'motor', motor_result_schemas(), output_path=FOUR_TASK_ROOT / 'motor_gate.json'
)
print({
    'motor_experiments':len(MOTOR_PLAN), 'assets':MOTOR_ASSET_STATUS,
    'local_period_nrmse':MOTOR_PERIOD_LOCAL['normalized_rmse'],
    'local_cssf_demo_target_status':MOTOR_FEW_FEM_LOCAL['methods']['CSSF-demo']['status'],
    'application_claim_status':MOTOR_GATE['claim_status'],
})

## EEG phase/syntax microstate optimization — native phase physics, exact/fidelity-gated syntax QUBO, QA response

N67-01…N67-21 implement the fourth active task. The toric coordinate is the continuous analytic EEG phase, never the discrete microstate label. The supplied Antonova synthetic notebook is treated as provenance material only: its unreproduced numbers, target-alignment problem, chi-square context-merging heuristic, and full-scale QUBO search/objective ambiguity are explicitly quarantined. Real-EEG, test-retest, independent replication, and live-QPU claims remain fail-closed until their required assets and confirmation records exist.

In [ ]:
# 79 — EEG N67-01…N67-21 experiment registry and external-asset gates
EEG_PLAN = [asdict(e) for e in EEG_EXPERIMENTS]
EEG_ASSETS = {
    'antonova_raw_or_derived_manifest': PROJECT_ROOT / 'data' / 'eeg_antonova_manifest.json',
    'protected_public_replication_manifest': PROJECT_ROOT / 'data' / 'eeg_public_replication_manifest.json',
    'preprocessing_protocol': PROJECT_ROOT / 'data' / 'eeg_preprocessing_protocol.json',
    'microstate_maps_or_clustering_manifest': PROJECT_ROOT / 'data' / 'eeg_microstate_maps_manifest.json',
}
EEG_ASSET_STATUS = {k: p.is_file() for k,p in EEG_ASSETS.items()}
write_json(FOUR_TASK_ROOT / 'eeg_plan.json', {
    'schema':'CSSF-QA-EEG-PHASE-SYNTAX-v51',
    'execution_complete':False,
    'experiments':EEG_PLAN,
    'asset_status':EEG_ASSET_STATUS,
    'layers':['continuous_toric_phase_identification','discrete_syntax_optimization','qa_trigonometrization_response'],
    'claim_status':'PENDING_REQUIRED_REAL_EEG_QUBO_QPU_AND_REPLICATION_EXECUTION',
})
print({'eeg_experiments':len(EEG_PLAN),'assets':EEG_ASSET_STATUS})

In [ ]:
# 80 — deterministic local EEG implementation checks; SYNTHETIC/STRUCTURAL ONLY, never application evidence
rng_eeg = np.random.default_rng(20260817)
t_eeg = np.arange(1024)
phase_period = 32
phase_demo = np.column_stack([
    2*np.pi*t_eeg/phase_period,
    2*np.pi*t_eeg/phase_period + 0.4,
    2*np.pi*t_eeg/phase_period - 0.7,
])
phase_demo = phase_demo + 0.03*rng_eeg.normal(size=phase_demo.shape)
EEG_PHASE_LOCAL = phase_periodicity_audit(phase_demo, max_lag=96, candidate_lag=phase_period)

labels_demo = ((np.sin(2*np.pi*t_eeg/phase_period) > 0).astype(int) + 2*(np.cos(2*np.pi*t_eeg/(2*phase_period)) > 0).astype(int))
signal_demo = np.column_stack([np.sin(phase_demo[:,0]), np.cos(phase_demo[:,1]), np.sin(phase_demo[:,2])])
EEG_X_ALIGNED, EEG_Y_ALIGNED, EEG_FEATURE_END, EEG_TARGET_TIME = aligned_lagged_features(
    signal_demo, labels_demo, window=8, horizon=1
)
EEG_ALIGNMENT_LOCAL = assert_prediction_alignment(EEG_FEATURE_END, EEG_TARGET_TIME, minimum_horizon=1)

gcv_rows = 160
gcv_phase = np.linspace(-np.pi, np.pi, gcv_rows, endpoint=False)
EEG_GCV_X = np.column_stack([np.ones(gcv_rows), np.exp(1j*gcv_phase), np.exp(-1j*gcv_phase)])
EEG_GCV_Y = 0.3 + 0.7*np.cos(gcv_phase) + 0.02*rng_eeg.normal(size=gcv_rows)
EEG_GCV_LOCAL = gcv_real_target_diagnostics(EEG_GCV_X, EEG_GCV_Y, n_lambdas=100, lam_range=(-8,8))

eps_demo = np.array([0.02,0.04,0.08,0.16,0.24])
resid_demo = 0.15*eps_demo**2.15
EEG_WEAK_COUPLING_LOCAL = weak_coupling_exponent_audit(eps_demo, resid_demo)

write_json(FOUR_TASK_ROOT / 'eeg_local_implementation_checks.json', {
    'schema':'CSSF-QA-EEG-LOCAL-IMPLEMENTATION-v51',
    'execution_complete':True,
    'synthetic_or_structural_only':True,
    'claim_passed':False,
    'application_evidence':False,
    'phase_periodicity':EEG_PHASE_LOCAL,
    'prediction_alignment':EEG_ALIGNMENT_LOCAL,
    'gcv_diagnostics':EEG_GCV_LOCAL,
    'weak_coupling_scaling':EEG_WEAK_COUPLING_LOCAL,
})
print({
    'eeg_local_checks':'PASS',
    'candidate_phase_lag':EEG_PHASE_LOCAL['candidate_lag'],
    'aligned_min_horizon':EEG_ALIGNMENT_LOCAL['minimum_observed_horizon'],
    'gcv_boundary_optimum':EEG_GCV_LOCAL['boundary_optimum'],
    'synthetic_is_application_evidence':False,
})

In [ ]:
# 81 — EEG syntax-QUBO bridge diagnostics; exact small-partition mathematics, not real-EEG evidence
TRAIN_COUNTS_DEMO = np.array([
    [90, 10,  5,  5],
    [75, 20, 10,  5],
    [10, 85, 10,  5],
    [15, 70, 10, 15],
    [10, 10, 75, 15],
    [10, 10, 20, 70],
], dtype=float)
CONF_COUNTS_DEMO = np.array([
    [45,  6,  3,  2],
    [38, 11,  4,  3],
    [ 7, 42,  5,  2],
    [ 8, 34,  6,  8],
    [ 5,  7, 37,  7],
    [ 6,  5, 10, 35],
], dtype=float)
PAIR_COST_DEMO = pairwise_merge_cost_matrix(TRAIN_COUNTS_DEMO)
PARTITIONS_DEMO = [
    np.array([0,0,1,1,2,2]),
    np.array([0,0,1,1,2,3]),
    np.array([0,1,2,3,4,5]),
    np.array([0,0,0,1,1,1]),
    np.array([0,1,1,2,2,3]),
    np.array([0,0,1,2,3,3]),
]
TRUE_NLL_DEMO = np.array([heldout_partition_nll(TRAIN_COUNTS_DEMO, CONF_COUNTS_DEMO, a) for a in PARTITIONS_DEMO])
PAIR_ENERGY_DEMO = np.array([pairwise_partition_energy(PAIR_COST_DEMO, a) for a in PARTITIONS_DEMO])
EEG_QUBO_FIDELITY_LOCAL = partition_bridge_fidelity(TRUE_NLL_DEMO, PAIR_ENERGY_DEMO, lower_is_better=True, top_k=3)
EEG_QUBO_GRAPH_LOCAL = qubo_assignment_graph_stats(PAIR_COST_DEMO, n_groups=4)

write_json(FOUR_TASK_ROOT / 'eeg_qubo_bridge_local_diagnostics.json', {
    'schema':'CSSF-QA-EEG-QUBO-BRIDGE-LOCAL-v51',
    'execution_complete':True,
    'synthetic_only':True,
    'claim_passed':False,
    'application_evidence':False,
    'bridge_fidelity':EEG_QUBO_FIDELITY_LOCAL,
    'logical_graph':EEG_QUBO_GRAPH_LOCAL,
    'boundary':'pairwise merge energy is a surrogate bridge unless independently validated against the true group-level held-out objective',
})
print({'eeg_qubo_local':'PASS','bridge_spearman':EEG_QUBO_FIDELITY_LOCAL['spearman'],'application_evidence':False})

In [ ]:
# 82 — EEG fail-closed result schemas and claim gate
EEG_LIVE_QPU_IDS = {'N67-15','N67-16','N67-21'}
EEG_REPLICATION_IDS = {'N67-20','N67-21'}
EEG_REAL_DATA_IDS = {
    'N67-02','N67-03','N67-04','N67-06','N67-08','N67-09','N67-10','N67-11',
    'N67-12','N67-13','N67-14','N67-15','N67-16','N67-17','N67-18','N67-19','N67-20','N67-21'
}
EEG_RESULT_SCHEMAS = [
    EEGResultSchema(
        e.experiment_id,
        requires_real_eeg=e.experiment_id in EEG_REAL_DATA_IDS,
        requires_live_qpu=e.experiment_id in EEG_LIVE_QPU_IDS,
        requires_replication=e.experiment_id in EEG_REPLICATION_IDS,
    )
    for e in EEG_EXPERIMENTS
]
EEG_GATE = eeg_fail_closed_gate(
    FOUR_TASK_ROOT / 'eeg',
    EEG_RESULT_SCHEMAS,
    output_path=FOUR_TASK_ROOT / 'eeg_gate.json',
)
print(json.dumps(EEG_GATE, indent=2))

In [ ]:
# 84 — conditional result fields; these strings are never treated as evidence
CONDITIONAL_RESULT_PLACEHOLDERS = conditional_result_placeholders()
write_json(FOUR_TASK_ROOT / 'conditional_result_placeholders.json', {
    'schema':'CSSF-QA-CONDITIONAL-PLACEHOLDERS-v60',
    'execution_complete':True,
    'not_evidence':True,
    'fields':CONDITIONAL_RESULT_PLACEHOLDERS,
})
print({'placeholder_fields':len(CONDITIONAL_RESULT_PLACEHOLDERS),'not_evidence':True})

In [ ]:
# 83 — synchronized four-task fail-closed gate
FOUR_TASK_GATE = four_task_gate(FOUR_TASK_ROOT, output_path=FOUR_TASK_ROOT / 'four_task_gate.json')
print(json.dumps(FOUR_TASK_GATE, indent=2))
if FOUR_TASK_GATE['claim_status'] != 'READY_FOR_CLAIM_REVIEW':
    print('New IEEE-33, motor, and EEG claims remain PENDING until required experiments and independent confirmation are complete.')

In [ ]:
# 85 — V00–V19 are the only claim authority; notebook booleans cannot override them
VERIFICATION_CONTEXT=VerificationContext.build(PROJECT_ROOT,EVIDENCE_ROOT,notebook_path=NOTEBOOK_PATH,mode='simulator',live_hardware_expected=True,execute_expensive_checks=True)
FIRST_PASS=run_all(VERIFICATION_CONTEXT,write=True)
if FIRST_PASS['reports']['V18']['status']==PASS:
    write_external_export_stub(EVIDENCE_ROOT,claim_grade=True,rows=[{'evidence_id':'endpoint:CSSF-full','verifier_id':'V18'}])
FINAL_VERIFICATION=run_all(VERIFICATION_CONTEXT,write=True)
print(json.dumps(FINAL_VERIFICATION['aggregate'],indent=2))
print(json.dumps({'evidence_matrix':EVIDENCE_MATRIX_GATE,'four_task':FOUR_TASK_GATE,'ieee33':IEEE33_GATE,'motor':MOTOR_GATE,'eeg':EEG_GATE},indent=2))
if not FINAL_VERIFICATION['aggregate']['claim_grade']:
    print('Claim remains LOCKED. Inspect V00–V19 reports; no external superiority table is authorized.')

SIMULATOR_EXPORT_READY = bool(FINAL_VERIFICATION['aggregate']['claim_grade'] and EVIDENCE_MATRIX_GATE['simulator_composition_claim_ready'] and FOUR_TASK_GATE['claim_status']=='READY_FOR_CLAIM_REVIEW' and EEG_GATE['claim_status']=='READY_FOR_CLAIM_REVIEW')
print({'simulator_export_ready':SIMULATOR_EXPORT_READY,'live_hardware_claim_unlocked':False})